# Chapter 2: Modern Optimization & Regularization (Fashion-MNIST with Adam & Dropout)

Welcome to **Chapter 2**!

In Chapter 1, we built our first neural network and scored 97.9% on handwritten digits (MNIST).
In this chapter, we step into a more challenging, realistic vision domain: **Fashion-MNIST** (Zalando Research).

Because clothing items (Shirts, Coats, T-shirts) share subtle textures and overlapping silhouettes, deeper architectures are needed ($784 \to 256 \to 128 \to 10$). Deeper networks bring two fundamental challenges:
1. **The Optimization Bottleneck**: How to achieve rapid, stable convergence using **Adam (Adaptive Moment Estimation)**.
2. **The Curse of Overfitting**: How to eliminate the generalization gap using **Inverted Dropout** and **L2 Weight Decay**.

---

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dataset import load_fashion_mnist, FASHION_MNIST_CLASSES, get_batches
from src.deep_model import DeepNeuralNetwork
from src.optimizers import SGDWithMomentum, Adam
from src.visualize import plot_confusion_matrix

print("Chapter 2 modules loaded successfully from repo root:", repo_root)

## 1. Load & Inspect the Fashion-MNIST Dataset

Fashion-MNIST contains 70,000 $28 \times 28$ grayscale images across 10 clothing classes:
`T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot`.

In [ ]:
data_path = repo_root / "data" / "fashion_mnist"
X_train, y_train, X_val, y_val, X_test, y_test = load_fashion_mnist(
    data_dir=str(data_path),
    val_size=10000,
    flatten=True,
    normalize=True,
)

print(f"Training set   : {X_train.shape[0]:,} images | Shape: {X_train.shape}")
print(f"Validation set : {X_val.shape[0]:,} images   | Shape: {X_val.shape}")
print(f"Test set       : {X_test.shape[0]:,} images   | Shape: {X_test.shape}")

In [ ]:
# Visualize sample garments from each class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c in range(10):
    idx = np.where(y_train == c)[0][0]
    ax = axes[c // 5, c % 5]
    ax.imshow(X_train[idx].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"{c}: {FASHION_MNIST_CLASSES[c]}", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Fashion-MNIST: 10 Clothing Categories", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 2. Inverted Dropout in Action

During training, **Dropout** silences a random fraction $p$ (e.g., 20%) of hidden neuron activations, scaling surviving neurons by $\frac{1}{1 - p}$.
During evaluation, dropout is disabled with **zero test-time overhead**.

Let us pass a sample clothing image through a layer and observe the activation vector in **Training Mode** vs **Evaluation Mode**:

In [ ]:
toy_net = DeepNeuralNetwork(layer_dims=[784, 64, 10], dropout_rate=0.25, seed=42)
sample_x = X_train[:1]

# 1. Forward in training mode (dropout active)
toy_net.train_mode()
toy_net.forward(sample_x)
train_activations = toy_net.cache["A1"][0]

# 2. Forward in evaluation mode (dropout inactive)
toy_net.eval_mode()
toy_net.forward(sample_x)
eval_activations = toy_net.cache["A1"][0]

plt.figure(figsize=(13, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(train_activations)), train_activations, color="#ef4444", alpha=0.85)
plt.title("Training Mode (Dropout p=0.25 Active)", fontsize=11, fontweight="bold")
plt.xlabel("Hidden Neuron Index")
plt.ylabel("Activation Value")
plt.grid(True, alpha=0.2)

plt.subplot(1, 2, 2)
plt.bar(range(len(eval_activations)), eval_activations, color="#3b82f6", alpha=0.85)
plt.title("Evaluation Mode (Dropout Inactive)", fontsize=11, fontweight="bold")
plt.xlabel("Hidden Neuron Index")
plt.ylabel("Activation Value")
plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

silenced = np.sum(train_activations == 0)
print(f"During training forward pass: {silenced}/{len(train_activations)} neurons were silenced ({silenced/len(train_activations)*100:.1f}%).")

## 3. Comparative Training Benchmark

We train two deep 3-layer networks ($784 \to 256 \to 128 \to 10$):
- **Model A**: SGD with Momentum (No Dropout, No Weight Decay)
- **Model B**: Adam Optimizer + Inverted Dropout ($p=0.2$) + L2 Weight Decay ($\lambda = 10^{-4}$)

In [ ]:
def train_deep_model(model, optimizer, epochs=5, batch_size=64):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for ep in range(1, epochs + 1):
        model.train_mode()
        for X_b, y_b in get_batches(X_train, y_train, batch_size=batch_size, shuffle=True):
            model.forward(X_b)
            grads = model.backward(y_b)
            for p_name in model.params.keys():
                model.params[p_name] = optimizer.step(p_name, model.params[p_name], grads[p_name])
        
        tr_l, tr_a = model.evaluate(X_train[:10000], y_train[:10000])
        v_l, v_a = model.evaluate(X_val, y_val)
        history["train_loss"].append(tr_l)
        history["train_acc"].append(tr_a)
        history["val_loss"].append(v_l)
        history["val_acc"].append(v_a)
        print(f"  Epoch {ep}/{epochs} | Train Loss: {tr_l:.4f} | Train Acc: {tr_a*100:.2f}% | Val Acc: {v_a*100:.2f}%")
    return history

layer_dims = [784, 256, 128, 10]

print("Training Model A (Deep MLP with SGD, No Regularization)...")
net_a = DeepNeuralNetwork(layer_dims=layer_dims, dropout_rate=0.0, weight_decay=0.0, seed=42)
hist_a = train_deep_model(net_a, SGDWithMomentum(lr=0.03, momentum=0.9), epochs=5)

print("\nTraining Model B (Deep MLP with Adam + Dropout + Weight Decay)...")
net_b = DeepNeuralNetwork(layer_dims=layer_dims, dropout_rate=0.2, weight_decay=1e-4, seed=42)
hist_b = train_deep_model(net_b, Adam(lr=0.001), epochs=5)

## 4. Comparing Learning Dynamics & The Generalization Gap

In [ ]:
epochs_range = range(1, 6)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Validation Accuracy
ax1.plot(epochs_range, [a * 100 for a in hist_a["val_acc"]], "o--", color="#dc2626", label="Model A (SGD) Val Acc")
ax1.plot(epochs_range, [b * 100 for b in hist_b["val_acc"]], "s-", color="#059669", linewidth=2, label="Model B (Adam+Dropout) Val Acc")
ax1.set_title("Validation Accuracy: SGD vs Adam on Fashion-MNIST", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy (%)")
ax1.grid(True, alpha=0.3)
ax1.legend()

# 2. Overfitting Gap (Train Acc - Val Acc)
gap_a = [t - v for t, v in zip(hist_a["train_acc"], hist_a["val_acc"])]
gap_b = [t - v for t, v in zip(hist_b["train_acc"], hist_b["val_acc"])]

ax2.plot(epochs_range, [g * 100 for g in gap_a], "o--", color="#dc2626", label="Model A Gap (No Regularization)")
ax2.plot(epochs_range, [g * 100 for g in gap_b], "s-", color="#059669", linewidth=2, label="Model B Gap (With Dropout)")
ax2.set_title("Generalization Gap: Overfitting Control", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Train Acc - Val Acc (%)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Unseen Test Set Evaluation & Confusion Matrix

In [ ]:
loss_a, acc_a = net_a.evaluate(X_test, y_test)
loss_b, acc_b = net_b.evaluate(X_test, y_test)

print(f"Model A (Unregularized SGD) Test Accuracy : {acc_a * 100:.2f}%")
print(f"Model B (Adam + Dropout)     Test Accuracy : {acc_b * 100:.2f}%")

# Compute confusion matrix for Model B
y_preds = net_b.predict(X_test)
cm = np.zeros((10, 10), dtype=int)
for true_c, pred_c in zip(y_test, y_preds):
    cm[true_c, pred_c] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(FASHION_MNIST_CLASSES, rotation=45, ha="right")
ax.set_yticklabels(FASHION_MNIST_CLASSES)
ax.set_xlabel("Predicted Clothing Item", fontweight="bold")
ax.set_ylabel("True Clothing Item", fontweight="bold")
ax.set_title("Model B Confusion Matrix on Test Garments", fontweight="bold", pad=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()